In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import numpy as np
from byte_util.util import single_to_double_float, all_sites, all_forcing_types
from byte_util.reaction import mineral_params, ssa_to_bsa, write_initial_cec
from min3p.input import InputFile
from min3p.output import write_min3p, read_min3p_sequence

rerun_erw = True

# Grid cell spacing
nz = 402
dz = 0.01

# Define the timesteps between output of transient data
transient_output_freq = {'hourly': 2, 'daily': 2, 'monthly': 1, 'longterm': 1}

# Get depth-varying intrinsic rate constants for CO2 production
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'

soil_pco2 = pd.read_csv(f'{s3_input_path}/soil_pco2.csv', index_col=0)
soil_phys = pd.read_parquet(f'{s3_input_path}/soil_physical_parameters.parquet')
soil_chem = pd.read_parquet(f'{s3_input_path}/soil_chemical_parameters.parquet')
root_params = pd.read_parquet(f'{s3_input_path}/root_water_parameters.parquet')

In [ ]:
# Load in forsterite application info
application_rate = 10  # t/ha
diameter = 50  # um
roughness = 20  # Assuming constant roughness

outfile = f'forsterite_{application_rate}tha_{diameter}um_{roughness}rf.csv'
forsterite = pd.read_csv(f'{s3_input_path}/{outfile}')

In [ ]:
from byte_util.util import start_date
# Calculate initial surface flux for each site and simulation type

columns = ['site', 'flux'] + all_forcing_types
init_forcing = pd.DataFrame(columns=columns, dtype=float)
init_forcing.set_index(['site', 'flux'], inplace=True)

for site in all_sites:
    forcing_file = f'{s3_input_path}/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Limit to 10-year simulation period
    end_date = pd.to_datetime(start_date) + pd.Timedelta(days=3651)
    met_forcing = met_forcing.loc[start_date:end_date, :]

    # Convert from mm/hr to m/s
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    # Convert transpiration from mm/hr to m/d
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24
    # Convert from m/d to 1/d
    met_forcing['transpiration_factor'] = met_forcing['transpiration_m.d'] / dz

    daily = met_forcing.resample('D').mean()
    monthly = met_forcing.resample('MS').mean()

    for col in ['surface_flux_m.s', 'transpiration_factor']:
        # Initial forcing value for spinup and longterm is the long-term mean
        init_forcing.loc[(site, col), 'spinup'] = met_forcing[col].mean()
        init_forcing.loc[(site, col), 'longterm'] = met_forcing[col].mean()

        # For hourly, daily, monthly, use the first value in the respective
        # time series. Subsequent values are changed by *.bcvs files
        init_forcing.loc[(site, col), 'hourly'] = met_forcing[col].values[0]
        init_forcing.loc[(site, col), 'daily'] = daily[col].values[0]
        init_forcing.loc[(site, col), 'monthly'] = monthly[col].values[0]

## Create MIN3P input file for longterm, monthly, daily, hourly (*.dat)

In [ ]:
for i, site in enumerate(all_sites):
    if not rerun_erw:
        continue

    sitepath = Path(f'../simulations/dual_perm_rxn/min3p_runs/{site}')
    spinpath = sitepath / 'spinup'

    for forcing_type in all_forcing_types:
        for scenario in ['ctrl', 'erw']:
            simpath = sitepath / f'{forcing_type}_{scenario}'
            shutil.rmtree(simpath, ignore_errors=True)
            simpath.mkdir(parents=True)

            infile = InputFile.load('spinup.dat', path=spinpath)

            # Change problem title
            gcp = infile.global_control_parameters
            label = 'Control' if scenario == 'ctrl' else 'ERW'
            gcp.problem_title = f'{label} simulation with {forcing_type} forcing'

            # Decrease maximum time step to 1 day for accuracy
            infile.time_step_control.maximum_time_step = 1.0

            # Adjust simulation end time and output times
            infile.time_step_control.final_time = 3650.
            oc = infile.output_control
            output_times = [float(t) for t in range(90, 3651, 90)]
            output_times.append(3650.)
            oc.output_of_spatial_data = output_times

            # # Adjust transient output frequency
            # infile.output_control.transient_output_interval = transient_output_freq[forcing_type]

            ## Adjust scenario-specific forcing
            # Adjust top boundary to initial rate
            bcvs = infile.boundary_conditions_vsflow
            top_boundary = bcvs.zones[0]
            infil_rate = init_forcing.loc[(site, 'surface_flux_m.s'), forcing_type]
            top_boundary.boundary_value = single_to_double_float(f'{infil_rate:0.3e}')

            # Adjust initial transpiration_factor
            for zone in infile.physical_parameters_vsflow.zones:
                if 'horizon' in zone.name:
                    rwu = zone.root_water_uptake
                    trans_factor = init_forcing.loc[(site, 'transpiration_factor'), forcing_type]
                    rwu.transpiration_factor = f'{trans_factor:0.6f}'

            # Adjust vsflow initial conditions
            infile.initial_conditions_vsflow.text = ("! Initial cond generated from spin-up\n"
                                                     "'read initial condition from file'\n")

            # If sim != longterm or spinup, add transient keywords
            if forcing_type != 'longterm':
                bcvs.text += "\n'transient boundary conditions'\n\n"
                infile.physical_parameters_vsflow.transient_transpiration = True

            # Adjust rt initial conditions
            icrt = infile.initial_conditions_reactive_transport
            zone_names = [z.name for z in icrt.zones]
            delete_zones = ['macropore chem', 'distribution layer 1 chem', 'distribution layer 2 chem',
                            'B horizon chem', 'C horizon chem']
            for zn in delete_zones:
                if zn in zone_names:
                    infile.delete_zone(zn, block_names=['initial_conditions_reactive_transport'])
            # Replace 'ph' and 'pco2' with 'free'
            for rec in icrt.zones[0].concentration_input.records:
                if 'ph' in rec.value():
                    rec.replace_content(f"{rec.value()[0]:0.2f}           'free'")
                elif 'pco2' in rec.value():
                    pco2_str = single_to_double_float(f'{rec.value()[0]:0.4e}')
                    rec.replace_content(f"{pco2_str}      'free'")
            icrt.read_cec_from_file = True
            icrt.read_initial_aqueous_component_concentrations_from_file = True
            icrt.read_initial_mineral_volume_fractions_from_file = True
            icrt.read_initial_mineral_areas_from_file = True

            # Adjust extent of the A horizon
            zone = icrt.zones[0]
            zone.extent_of_zone = f'0.0 1.0  0.0 1.0  0.00 4.01'

            # Adjust concentration of top BC to 1e-3
            bcrt = infile.boundary_conditions_reactive_transport  # Get BC block
            comp = infile.geochemical_system.components
            top_conc = bcrt.zones[0].concentration_input  # Get concentrations of top BC
            top_conc.records[comp.index('psi01')].replace_content("1.0000d-3      'free'")

            infile.save(simpath / f'{forcing_type}.dat')

## Copy (and modify) files to specify initial conditions

In [ ]:
from byte_util.reaction import dualperm_co2_factors
import fsspec

# Load in and scale CO2 respiration
soil_resp_calc = 'GB94'
database_rate = 1e-13
co2_scaling_factor = np.array(list(dualperm_co2_factors.values()), dtype=np.float64)
k_co2_path = f'{s3_input_path}/soilco2production_profiles_{soil_resp_calc}.npy'

with fsspec.open(k_co2_path, 'rb') as f:
    k_co2 = np.load(f)
k_co2 *= co2_scaling_factor[:, np.newaxis]

# Add distribution layer
k_co2 = np.insert(k_co2, 0, np.zeros(k_co2.shape[0]), axis=1)

In [ ]:
for i, site in enumerate(all_sites):
    if not rerun_erw:
        continue

    sitepath = Path(f'../simulations/dual_perm_rxn/min3p_runs/{site}')
    spinpath = sitepath / 'spinup'

    # Skip if spinup isn't finished
    first_gsp = spinpath / 'spinup_1.gsp'
    if not first_gsp.exists():
        continue

    if site == 'Yolo':
        horizons = ['A', 'C']
    elif site == 'Pullman':
        horizons = ['A', 'B']
    else:
        horizons = ['A', 'B', 'C']

    for forcing_type in all_forcing_types:
        for scenario in ['ctrl', 'erw']:
            simpath = sitepath / f'{forcing_type}_{scenario}'

            ### Copy over rld, bcvs and soi from met_forcing_transport simulations
            shutil.copy(spinpath / 'spinup.rld', simpath / f'{forcing_type}.rld')
            for extension in ['soi', 'bcvs']:
                if forcing_type == 'longterm':
                    continue
                transport_basepath = Path('../simulations/met_forcing_transport/min3p_runs')
                trans_file_base = transport_basepath / site / forcing_type / f'{forcing_type}.{extension}'
                trans_file_sim = simpath / f'{forcing_type}.{extension}'
                shutil.copy(trans_file_base, trans_file_sim)

            ### Copy over final gsp file
            gsp_files = [f for f in spinpath.glob('*.gsp')]
            gsp_files.sort(key=lambda x: int(x.stem.split('_')[-1]))
            shutil.copy(gsp_files[-1], simpath / f'{forcing_type}.ivs')

            ### Create initial CEC file
            cec_columns = ['x', 'y', 'z', 'cec', 'rho']
            cec = write_initial_cec(Path(''), cec=soil_chem.loc[(site, horizons), 'CEC_meq_100g'].values,
                                    bulk_density=soil_phys.loc[(site, horizons), 'rho_g.cm3'].values,
                                    elev_top=4.0 - soil_phys.loc[(site, horizons), 'top_m'].values,
                                    elev_bot=4.0 - soil_phys.loc[(site, horizons), 'bottom_m'].values,
                                    nz=nz, return_array=True, write_to_file=False)
            init_cec = np.tile(cec[:, np.newaxis, :], (1, 2, 1))  # Reshape to ncol, ny, nz
            init_cec[1, 1, :] = 1.0  # Set matrix y-coordinate
            # Set macropore CEC to either B or C horizon
            macropore_cec_hzn = 'B' if site == 'Pullman' else 'C'
            init_cec[3, 0, :] = soil_chem.loc[(site, macropore_cec_hzn), 'CEC_meq_100g']
            # Set distribution layer (top layer) to 0 CEC
            init_cec[3, :, -1] = 0.0
            cec_reshaped = np.reshape(init_cec, (init_cec.shape[0], nz*2), order='F')
            write_min3p(cec_reshaped, f'{forcing_type}.cec', cec_columns, folder=simpath,
                        prefix=forcing_type, label='initial CEC')

            ### Copy over initial aqueous concentrations, modifying tracers
            gst, gst_cols, timesteps = read_min3p_sequence(spinpath / 'spinup_1.gst')
            aqt = gst[-1]
            # Set first tracer (psi01) to 1e-1 in the shallow macropores (y=0, z=3.7 - 4.0 m)
            aqt[gst_cols.index('psi01'), :, :] = 1e-3
            aqt[gst_cols.index('psi01'), 0, int(3.7/dz)+1:int(4.0/dz)+1] = 1e-1
            # Set second tracer (psi02) to 1e-1 in the shallow matrix (y=1, z=3.7 - 4.0 m)
            aqt[gst_cols.index('psi02'), :, :] = 1e-3
            aqt[gst_cols.index('psi02'), 1, int(3.7/dz)+1:int(4.0/dz)+1] = 1e-1
            # Write to file
            aqtfile = simpath / f'{forcing_type}.aqt'
            aqt_reshaped = np.reshape(aqt, (len(gst_cols), -1), order='F')
            write_min3p(aqt_reshaped, aqtfile, gst_cols, prefix=forcing_type, label='initial aqueous concentrations')

            ### Copy over initial mineral volume fractions
            gsv, gsv_cols, timesteps = read_min3p_sequence(spinpath / 'spinup_1.gsv')
            minv = gsv[-1]
            if scenario == 'erw':
                forst_i = gsv_cols.index('forst-ph')
                volfrac = forsterite['vol_frac_in_soil'].values.copy()
                volfrac[volfrac == 0] = 1e-10  # Replace 0 with 1e-10 (minimum frac for stability)
                # Assume forsterite only applied to soil matrix
                minv[forst_i, 1, -forsterite.shape[0]:] = volfrac[::-1]/0.95  # dividing by 0.95 (matrix volume)
            minfile = simpath / f'{forcing_type}.min'
            minv_reshaped = np.reshape(minv, (len(gsv_cols), -1), order='F')
            write_min3p(minv, minfile, gsv_cols, prefix=forcing_type, label='phi_i, T = initial')

            ### Update mineral surface areas
            minerals = [c for c in gsv_cols if c not in ['x', 'y', 'z', 'porosity']]
            surface_areas = np.zeros((len(minerals)+3, 2, minv.shape[-1]), dtype=float)
            # Add in y
            surface_areas[1, 0, :] = 0.0
            surface_areas[1, 1, :] = 1.0
            # Add in z column
            surface_areas[2, :] = minv[2, :]
            for i, mineral in enumerate(minerals):
                if mineral == 'co2_resp':
                    site_idx = all_sites.index(site)
                    surface_areas[i+3, 0, :] = np.zeros(nz, dtype=float)
                    surface_areas[i+3, 1, :] = np.flip(k_co2[site_idx])/database_rate
                else:
                    for d in range(2):
                        surface_areas[i+3, d, :] = ssa_to_bsa(vol_frac=minv[i+3, d, :], ssa=mineral_params[mineral]['ssa'],
                                                              density=mineral_params[mineral]['density'])
            surffile = simpath / f'{forcing_type}.surf'
            surf_reshaped = np.reshape(surface_areas, (len(minerals)+3, -1), order='F')
            write_min3p(surface_areas, surffile, ['x', 'y', 'z'] + minerals,
                        prefix=forcing_type, label='bsa_i (m2/L), T = initial')